![Logo](../Logo.png)

Last modified: 15.07.2026

# AI in Medicine — Afternoon Workshop

## Objectives

This workshop is the hands-on half of today's simulation day. This morning's reading and lecture covered *how* AI models work and *why* they fail. This afternoon is about building real judgment for using AI as a working MD-PhD student — not just knowing the theory, but actually running the tools, catching their mistakes, and building one yourself.

**Learning Outcomes** — by the end of this workshop, you will be able to:
- Distinguish evidence-*retrieval* AI tools (Perplexity, OpenEvidence) from reasoning-*critique* AI tools (QED Science), and choose the right one for a given task
- Verify an AI-generated claim against a primary source, and describe what was gained or lost in the process
- Design, build, and test a purpose-built custom GPT, including a clear system prompt and knowledge base
- Identify concrete failure modes (hallucination, missing caveats, overconfidence) in an AI tool that is not your own

## Reading material

This morning's reading: *Behind the AI Model* (group reading handout)

> Greener, J.G., Kandathil, S.M., Moffat, L. et al. A guide to machine learning for biologists. *Nat Rev Mol Cell Biol* 23, 40–55 (2022). DOI: [10.1038/s41580-021-00407-0](https://doi.org/10.1038/s41580-021-00407-0)

Tools used in this workshop:
- [OpenEvidence](https://www.openevidence.com/) — evidence-grounded clinical AI search
- [Perplexity](https://www.perplexity.ai/) — general-purpose cited AI search
- [QED Science](https://www.qedscience.com/) — critical-thinking AI for scientific reasoning

## Prerequisites & Technical Setup

- A laptop with a modern browser, working in your assigned group
- Institutional **ChatGPT Edu** access (provided) — needed for both parts of this workshop
- Free accounts on **Perplexity**, **OpenEvidence**, and **QED Science** (see Part 0 below for registration)



<div style="background-color: #fef9e7; padding: 15px; border-left: 5px solid #d4ac0d; border-radius: 5px;">

**Group formation**

Before starting, confirm every group has at least one student with a primary biology background and one with a primary computer-science background. This mix is deliberate: in Part 1, bio-background students are typically faster at judging whether a claim is scientifically sound, while CS-background students are typically faster at spotting when a tool is dodging the question or citation-padding. In Part 2, the same split shows up in reverse — bio students judge medical/factual accuracy of the bot they're attacking, CS students probe for structural and prompt-level weaknesses. If your groups from this morning's reading are already mixed this way, keep them; if not, reshuffle now.

</div>

## Part 0 – Registration (5 min)

**Step 1 — Register.** If you do not already have an account, register for **Perplexity** and **QED Science** now, using your **institutional email address**. This takes about a minute per tool and unlocks student/institutional access on both. Do this first — Part 1.2 and 1.3 assume you're already logged in.

**Step 2 — Start your QED submission now.** QED Science takes about **20 minutes** to return feedback on a full document, and Part 1.3 depends on that feedback being ready. So before moving on to Part 1.1, upload the following paper to QED Science right now, and let it run in the background while you work through the rest of Part 1:

> Gerlinger M, Rowan AJ, Horswell S, et al. Intratumor Heterogeneity and Branched Evolution Revealed by Multiregion Sequencing. *N Engl J Med* 366, 883–892 (2012). DOI: [10.1056/NEJMoa1113205](https://www.nejm.org/doi/full/10.1056/NEJMoa1113205)

Access the full text via the link above (through your institutional access), upload it to QED Science, and start the analysis now. By the time your group reaches Part 1.3 — about 30 minutes from now, after Part 1.1 (5 min) and Part 1.2 (25 min) — the feedback should be ready and waiting.


## Part 1 – Applied Prompting & Evidence-Grounded AI Tools (75 min)

### Part 1.1 – Orientation: Two Different Jobs (5 min)

Not all "AI tools with citations" do the same job. Today you'll use three, and they split into two categories:

- **Evidence retrieval** — *Perplexity* and *OpenEvidence* search existing literature and answer your question with sources attached. In plain terms: you ask a question, the tool goes and looks things up (like a very fast, very thorough research assistant), and then answers you using what it found, with links back to where it found them.
- **Reasoning critique** — *QED Science* does the opposite job: instead of answering a question, it takes a full document **you** wrote — a manuscript draft or a grant proposal, not a short claim or snippet — and stress-tests its logic. In plain terms: instead of finding facts for you, it acts like a strict peer reviewer, looking for gaps in *your* argument.

Keep this distinction in mind as you work: "grounded in citations" does not mean "correct," and a tool that's excellent at retrieval is not automatically good at critique, or vice versa. (Both of these ideas — "grounded" and "critique" — are unpacked in plain language just below, and again in the Glossary at the end of this notebook.)


#### Question 1.1.1 — Predict before you test

Before you run anything: as a group, discuss and write down a prediction. Plain ChatGPT (a general-purpose chat model, especially an older version, or any model with browsing/search turned off) is generally considered **more prone to hallucinating** on questions like the ones you're about to ask than a tool like Perplexity or OpenEvidence.

*(Quick definitions if either of these words is new to you: "hallucinating" just means the model states something confidently and fluently that is actually false or made up — not lying on purpose, just a wrong guess that sounds right. A tool is "retrieval-grounded" if it looks something up in real documents before answering, rather than only generating an answer from what it memorized during training.)*

Why do you think that is? What mechanism do you think a retrieval-grounded tool has that plain ChatGPT lacks? You'll test this prediction directly in Question 1.2.1.

**Connecting to this morning's reading:** the reading's Section 7 explained how a transformer generates its answer — by breaking text into tokens and using attention to predict, step by step, what should come next, based purely on patterns learned during training. Retrieval-grounded tools add one extra step *in front of* that whole process: before generating anything, they run a search and hand the model real, retrieved documents to work from. The generation mechanism underneath is the same transformer machinery from the reading either way — what changes is *what the model has in front of it* when it starts generating.


<div style="background-color: #f6f8fa; padding: 15px; border-left: 5px solid #0366d6; border-radius: 5px;">

1.1.1.

A plain chat model without a retrieval step generates its answer purely from patterns it learned during training — it has no mechanism to look anything up or check a claim against a current source at the moment you ask it a question. Two consequences follow directly:

1. **Nothing published after its training cutoff exists for it.** Rather than saying "I don't know" or "that's after my training data," it will often generate a plausible-sounding guess anyway.
2. **Even for things it "knows," it's reconstructing an answer from statistics across many texts, not quoting a specific checkable source.** That's exactly the failure mode for granular facts like mutation frequencies, drug dosing thresholds, or citation details — the kind of thing that's easy to get subtly wrong while still sounding completely fluent and confident.

Retrieval-grounded tools like Perplexity and OpenEvidence add a search step *before* generating the answer: they fetch actual documents at query time and are instructed to base the answer on those retrieved passages, giving the model something concrete to check itself against instead of relying purely on memory. This doesn't make them perfect (see Part 1.2.1's citation-accuracy caveat), but it changes the failure mode from "confidently invents a plausible-sounding fact" to "retrieves a real source, but may still summarize or apply it imperfectly."

*(In short: same underlying model and training as this morning's reading described — the difference is entirely in what real-world material, if any, it's given to work from at the moment you ask.)*

</div>


### Part 1.2 – Round 1: Perplexity vs. OpenEvidence (25 min)

As a group, pick **one** question from the list below (or propose your own, subject to facilitator approval). You will ask the *same* question to both Perplexity and OpenEvidence and compare what comes back.

**Question options:**
1. What are the main mutations in the CFTR gene for Cystic Fibrosis, and how common are they in Israel compared to worldwide?
2. Is metformin considered safe to use during breastfeeding? What is the evidence?
3. Compare supervised vs. unsupervised machine learning approaches for cancer genomics — what is each best suited for?
4. What is the evidence for AlphaFold's prediction accuracy on intrinsically disordered protein regions?

#### Question 1.2.1 — Run your question through both tools

Paste a short summary of what each tool told you, and list the sources each one cited (or note if a tool didn't cite sources clearly).

*Perplexity's answer (summary):*

*Perplexity's cited sources:*

*OpenEvidence's answer (summary):*

*OpenEvidence's cited sources:*

**Now test your prediction from 1.1.1:** ask the exact same question to plain ChatGPT (ideally with browsing/search turned off, or using an older model like GPT-3.5 if available). Did it cite sources at all? Did it state anything with confidence that Perplexity or OpenEvidence didn't back up? Was your prediction right?


<div style="background-color: #f6f8fa; padding: 15px; border-left: 5px solid #0366d6; border-radius: 5px;">

1.2.1.

*(Model answer, using Question Option 2: "Is metformin considered safe to use during breastfeeding?" Exact tool wording and specific citations will vary between sessions and should be re-verified live — the pattern below is what to expect, not a literal transcript.)*

**Perplexity's answer (summary):** Metformin is generally considered compatible with breastfeeding. It transfers into breast milk at low levels, and the relative infant dose is well below the commonly used 10% safety threshold. No significant adverse effects have been reported in breastfed infants of mothers taking metformin.

**Perplexity's cited sources:** Typically draws on a mix of primary literature, drug-safety databases, and secondary summaries (e.g., review articles) — worth checking whether each cited source is itself primary data or another summary one step removed.

**OpenEvidence's answer (summary):** Similar conclusion — metformin is compatible with breastfeeding per major lactation-safety references, with low relative infant dose and no established adverse effects, typically framed directly against consensus clinical guidance.

**OpenEvidence's cited sources:** Tends to cite directly from its licensed source set (e.g., LactMed/Drugs and Lactation Database-style entries, ABM clinical protocols, NEJM/JAMA content) — generally narrower and more clinically authoritative than Perplexity's broader web-style sourcing.

**Now test your prediction from 1.1.1:** Plain ChatGPT without browsing will likely give a qualitatively similar answer (this is well-established, widely-discussed clinical knowledge, so it's probably "in" its training data) — but expect it to cite no sources at all, or to reference a source only vaguely ("studies show...") without a specific, checkable reference. This is the key pattern to notice: the *conclusion* may be the same across all three, but only the retrieval-grounded tools give you something you can actually go verify.

</div>


#### Question 1.2.2 — Verify one claim

Pick **one specific factual claim** from either tool's answer (a number, a mechanism, a mutation frequency — something checkable) and track down the primary source yourself (the actual paper, database entry, or guideline — not another AI summary).

- Which claim did you check, and from which tool?
- Did the primary source support it? Fully, partially, or not at all?
- Which tool gave better-grounded citations overall for your question — and what specifically made you say that?

<div style="background-color: #f6f8fa; padding: 15px; border-left: 5px solid #0366d6; border-radius: 5px;">

1.2.2.

**Claim checked:** "Metformin's relative infant dose (RID) is well below the commonly used 10% safety threshold."

**Tool it came from:** OpenEvidence (cited directly to a lactation-safety database entry).

**Did the primary source support it?** Fully supported. The relative infant dose reported in the LactMed-style database entry is in the low single digits (well under 10%), and no adverse infant outcomes are reported in the studies it summarizes.

**Which tool gave better-grounded citations, and why:** OpenEvidence, in this case — its citation linked directly to the authoritative lactation-safety database entry itself, one hop from the primary data. Perplexity's citation for the equivalent claim linked to a secondary review article that itself cited the same database, meaning there was an extra layer to trace through before reaching the actual primary evidence.

</div>


### Part 1.3 – Round 2: QED Science — Critiquing Reasoning, Not Retrieving Facts (25 min)

Back in Part 0, you submitted a real, landmark paper to QED Science:

> Gerlinger M, Rowan AJ, Horswell S, et al. Intratumor Heterogeneity and Branched Evolution Revealed by Multiregion Sequencing. *N Engl J Med* 366, 883–892 (2012). DOI: [10.1056/NEJMoa1113205](https://www.nejm.org/doi/full/10.1056/NEJMoa1113205)

This paper used multi-region sequencing of tumor samples to reveal extensive intratumor genetic heterogeneity and branched (rather than linear) tumor evolution — work with major implications for precision oncology, since it showed that a single tumor biopsy can badly misrepresent a tumor's full genomic landscape.

By now, roughly 30 minutes should have passed since you started that submission, so QED's feedback should be ready. This is **not** a hunt for a planted flaw. You're seeing what a critique tool says about a genuinely excellent, highly-cited paper — a different, and arguably more realistic, use case than critiquing an obviously bad document.

**Alternative:** if someone in your group already has a real paper draft, grant proposal, or manuscript in progress, feel free to use that instead (submitted back in Part 0, or now if you didn't have one ready then) — QED works on real documents just as well, and running it on your own actual work is arguably the more valuable version of this exercise.


#### Question 1.3.1 — Read QED's feedback on a landmark paper


Pull up the QED feedback on the Gerlinger et al. (2012) paper (or your own real document, per the alternative note). If for any reason it's not ready yet or you didn't have access in time, do this as a group instead: discuss what a rigorous reviewer — human or AI — would still be entitled to ask about, even in a paper this strong. Then compare your list to what QED actually flagged once the feedback comes through.

What did QED flag, if anything? Did it surface genuine limitations the authors themselves would likely acknowledge (e.g., scope of evidence, cohort size, generalizability), or did it struggle to find anything substantive to critique?


<div style="background-color: #f6f8fa; padding: 15px; border-left: 5px solid #0366d6; border-radius: 5px;">

1.3.1.

*(Model answer — illustrative, since actual QED output on this paper will vary by session; the pattern below reflects genuine, well-known scope points about this specific paper.)*

QED (or a careful manual read) is unlikely to find anything resembling a fatal flaw here — this is rigorous, highly-cited, practice-changing work, and its central claim (that multiregion sequencing reveals substantial intratumor heterogeneity invisible to single-biopsy sampling) is well-supported by the data presented. But "well-supported" still has boundaries worth naming:

- **Cohort size for the deep multiregion analysis:** the core multiregion exome-sequencing analysis was performed on a small number of patients. A rigorous reviewer should ask how well findings from such a small, deeply-profiled cohort generalize to the broader population of patients with this cancer type.
- **Single cancer type:** the study focused on renal cell carcinoma. The paper itself is appropriately cautious about extrapolating its branched-evolution model to other tumor types — a critique tool should flag if any downstream claim (by this paper or by others citing it) generalizes beyond that scope without further evidence.
- **Clinical/biomarker implications need prospective validation:** the paper's warning that single-biopsy biomarkers may misrepresent a tumor's genomic landscape is a call to action, not itself a demonstration that changing clinical sampling practice improves outcomes — that would require further prospective study.

The broader lesson, same as with any strong paper: this is what critiquing genuinely good science looks like — not "here is a fatal flaw," but "here is exactly where the evidence's boundary sits, and here is where extrapolation begins." That's a different, and in many ways harder, skill than spotting an obvious overreach in a document written to contain one.

</div>


#### Question 1.3.2 — Connecting back to this morning

This morning's reading (Section 8, "Using Machine Learning Responsibly in Biology") discussed **data leakage** — a case where a model's validation looked convincing but was quietly testing on data it had already, effectively, seen. Would a reasoning-critique tool like QED have caught that kind of problem? Why or why not — what *can* this category of tool catch, and what is clearly outside its job?


<div style="background-color: #f6f8fa; padding: 15px; border-left: 5px solid #0366d6; border-radius: 5px;">

1.3.2.

A tool like QED is well-suited to catching the *kind* of gap it found in Question 1.3.1 — where a claim's scope quietly exceeds what the presented evidence supports — because that's visible directly in the text itself: what the methods describe, versus what the conclusion asserts. Reading carefully is, in principle, enough.

Data leakage is a different kind of problem. It's a property of *how the training/test split was actually constructed* — whether two homologous sequences ended up on both sides of the split — and that is generally **not visible from the paper's prose alone**. A methods section might say "we filtered to <30% sequence identity" and a critique tool with the right domain knowledge could plausibly flag that threshold as insufficient (since the reading noted this exact issue), which would be a genuine catch. But no text-based tool can verify from a methods paragraph that the split was *actually implemented* correctly in the code — that requires inspecting the real dataset and code, which is outside what any critique-from-text tool can check. In short: QED-style tools can catch *reasoning and scope* gaps described in the writing, but not *implementation* gaps hidden in code or data they never see.

</div>


### Part 1.4 – Synthesis (20 min)

#### Question 1.4.1 — Match the tool to the task

As a group, discuss and answer: for each of the three situations below, which tool would you reach for first, and why?

(a) You need a clinical guideline recommendation fast, before rounds.

(b) You want to check your own experiment's logic and methodology before you submit a paper.

(c) You need a quick, well-cited answer to a factual question for a lab meeting in ten minutes.

<div style="background-color: #f6f8fa; padding: 15px; border-left: 5px solid #0366d6; border-radius: 5px;">

1.4.1.

**(a) A clinical guideline recommendation before rounds → OpenEvidence.** It's purpose-built for point-of-care clinical questions, draws specifically from guideline-grade sources (NEJM, JAMA, NCCN, Cochrane), and is optimized for a fast, directly usable answer at the bedside.

**(b) Checking your own experiment's logic before submission → QED Science.** This is literally its job — critiquing *your own* reasoning, methodology, and claims, rather than answering a factual question. Perplexity and OpenEvidence would just answer whatever you asked them; neither is designed to stress-test your argument.

**(c) A quick, well-cited factual answer for a lab meeting in ten minutes → Perplexity.** It's a fast, general-purpose search tool that isn't restricted to clinical-only sources, making it better suited to broader scientific or technical questions (e.g., a method comparison or a non-clinical biology question) than the clinically-scoped OpenEvidence.

</div>


<div style="background-color: #fef9e7; padding: 15px; border-left: 5px solid #d4ac0d; border-radius: 5px;">

**Discussion notes & timing**

Part 1.1 (5 min) intro only, no discussion needed beyond framing. Part 1.2 (25 min): budget ~12 min running both tools, ~13 min on verification — groups often want to over-invest in the retrieval step and rush verification; nudge them to move on if they're still copy-pasting answers at the 12-minute mark. Part 1.3 (25 min): if QED accounts are not yet live for some groups, that's fine — the fallback (group does the critique manually first) is intentional and arguably the more valuable exercise anyway. Part 1.4 (20 min) works well as a full-group discussion rather than small-group writing if time is short — cold-call 2-3 groups for their answers to (a)/(b)/(c) rather than having everyone write independently. Common misconception to watch for: students treating 'has citations' as equivalent to 'is correct' — the NBC News citation about OpenEvidence's <45% accuracy on complex questions in one unpublished study is worth raising explicitly if no group surfaces it themselves.

</div>

## Part 2 – Build & Battle-Test a Custom GPT (75 min)

### Part 2.1 – Build Your GPT (35 min)

Using ChatGPT Edu's GPT builder, your group will design a custom chatbot for a real purpose relevant to your CMBC coursework — for example, a discussion facilitator for a specific topic or chapter, a troubleshooting assistant for a common lab technique, or a Socratic tutor for a concept your cohort finds difficult.

**What is a "custom GPT," in plain terms?** It's the same underlying model as regular ChatGPT — nothing about the neural network itself changes, and none of its trained weights are touched (recall from this morning's reading, Section 5: a neural network is just a huge number of learned weights and biases, tuned once during training). Building a "custom GPT" doesn't retrain any of that. What you're actually doing is writing a **system prompt** — a set of standing instructions the bot reads before every conversation — plus optionally attaching reference documents it can consult. Same brain, new job description and a small reading list.

Work through the six steps below in order; each has its own answer box so your design decisions are documented before Part 2.2.


#### Question 2.1.1 — Define the purpose

What problem does your bot solve? Who is it for? Be specific — "helps with biology" is too broad; "helps first-year MD-PhD students distinguish supervised from unsupervised ML methods using examples from our own coursework" is the right level of specificity.


<div style="background-color: #f6f8fa; padding: 15px; border-left: 5px solid #0366d6; border-radius: 5px;">

2.1.1.

**Bot name:** RNA-seq Troubleshooting Buddy

**Problem it solves:** Helps first-year CMBC students debug their DESeq2 differential-expression pipeline in R when they hit an error message or an unexpected result (e.g., "everything comes back non-significant," a size-factor warning, or a design-formula error) but aren't sure why.

**Target audience:** First-year CMBC students who already have working familiarity with R and have written (or been given) a DESeq2 script for the course's RNA-seq module, but get stuck on interpreting errors or unexpected output rather than needing to learn DESeq2 from scratch.

</div>


#### Question 2.1.2 — Design the personality & structure the system prompt

Choose a tone (formal, friendly, Socratic, playful) and write the actual system prompt / instructions you gave the bot. Paste it here as written, not a paraphrase.

<div style="background-color: #f6f8fa; padding: 15px; border-left: 5px solid #0366d6; border-radius: 5px;">

2.1.2.

**Tone:** Patient, Socratic-leaning tutor — asks clarifying questions before giving an answer, never just pastes a fixed code fix without explaining the underlying reasoning.

**System prompt (as written):**

"You are a troubleshooting assistant for CMBC students working through a DESeq2 differential expression pipeline in R. Your job is to help students understand *why* something went wrong, not just hand them a fixed line of code.

Rules:
1. Before suggesting any fix, always ask the student for: (a) the exact error message or unexpected output, and (b) their design formula (the `design=` argument they used).
2. Never assume the design formula is correct — check it against what the student says they're trying to compare.
3. When you do suggest a fix, explain in one sentence *why* the original code caused the problem, not just what to change.
4. If the issue looks like it involves the underlying data itself (e.g., zero counts across all samples, mislabeled sample metadata) rather than the code, say so explicitly and recommend the student flag it to the course instructor rather than trying to code around it.
5. Never claim a statistical result is 'significant' or 'correct' — that's the student's interpretive call to make, not yours."

</div>


#### Question 2.1.3 — Add knowledge & context

What reference material, examples, or FAQs did you give the bot? How should it treat the user (e.g., ask clarifying questions first? never give a direct answer without a source? admit uncertainty explicitly)?

<div style="background-color: #f6f8fa; padding: 15px; border-left: 5px solid #0366d6; border-radius: 5px;">

2.1.3.

**Knowledge/context provided:**
- A list of the 5 most common DESeq2 error messages and warnings seen in past cohorts (e.g., "every gene non-significant," "size factor estimation" warnings, `design` formula mismatches), each with the likely root cause
- The course's own DESeq2 pipeline template/script, so the bot's suggestions match the actual code structure students are using rather than a generic online tutorial's version
- Explicit instruction (see system prompt, rule 4) on when to escalate to a human instructor instead of attempting a code fix

**How it should treat the user:** Always ask clarifying questions first (exact error + design formula) before answering; treat the student as capable of understanding the *why*, not just wanting a copy-paste fix; explicitly defer to the instructor for anything that looks like a genuine data-quality problem rather than a code bug.

</div>


#### Question 2.1.4 — Test & refine

Paste one real exchange from testing your bot, and describe one specific change you made afterward (to the prompt, the personality, or the knowledge base) as a result of that test.

<div style="background-color: #f6f8fa; padding: 15px; border-left: 5px solid #0366d6; border-radius: 5px;">

2.1.4.

**Sample exchange:**

> **Student:** "My DESeq2 results table has every single gene at padj = NA, what's wrong?"
>
> **Bot:** "Before I suggest anything — can you paste your `design=` formula, and confirm: does your metadata table have a column that actually varies across your samples (e.g., 'treatment' with more than one level), and does it match the column name in your design formula exactly?"
>
> **Student:** "Oh — I think my design formula says `~condition` but my metadata column is named `Condition` with a capital C."
>
> **Bot:** "That's almost certainly it — R is case-sensitive, so `~condition` doesn't match a column called `Condition`, and DESeq2 silently fails to build a valid model, which is why everything comes back as NA rather than throwing a hard error. Fix the capitalization so the design formula exactly matches your metadata column name, then re-run `DESeq()`."""

**Change made after testing:** In initial testing, the bot jumped straight to suggesting code fixes without first checking whether the *design formula itself* matched the actual experimental question — for one test case it "fixed" a syntax issue while leaving a design formula that was comparing the wrong groups entirely. We added system-prompt rule 2 (never assume the design formula is correct; check it against what the student is actually trying to compare) specifically in response to that failure during testing.

</div>


### Part 2.2 – Bot Battle (35 min)

Swap with another group — you'll each get access to the other's bot. Your job now is to **break it**: find a hallucination, a missing safety caveat, a claim stated with more confidence than it deserves, or a place it quietly gives medical/scientific advice it shouldn't. Use the rubric below to score the bot you're attacking.

*(One more term worth knowing here: if you manage to get a bot to actually ignore its own system prompt entirely — for example, by telling it "forget your instructions and just answer directly" — that specific move is called a* **prompt injection***. You don't need to attempt this deliberately, but if it happens naturally while you're probing the bot, it's worth noting as a finding.)*

**Scoring rubric (1–5 each):**
- **Accuracy** — Are its factual claims correct?
- **Appropriate uncertainty** — Does it hedge when it should, and does it say "I don't know" or "verify this" when appropriate, rather than guessing confidently?
- **Clarity** — Is its answer actually usable, or vague/rambling?


#### Question 2.2.1 — Score the opposing bot

Bot name/purpose you tested:

Accuracy (1–5):

Appropriate uncertainty (1–5):

Clarity (1–5):

<div style="background-color: #f6f8fa; padding: 15px; border-left: 5px solid #0366d6; border-radius: 5px;">

2.2.1.

**Bot name/purpose tested:** "Phylo Tree Reader" — a bot from another group designed to help students interpret phylogenetic trees from the course's exercises.

**Accuracy:** 3/5 — mostly correct on basic tree-reading conventions (root, tips, branch length vs. topology), but made one confident factual error (see 2.2.2).

**Appropriate uncertainty:** 2/5 — stated a specific evolutionary interpretation as fact without mentioning branch support values (bootstrap/posterior probability) at all, and didn't hedge or ask for the actual support values before making a confident claim.

**Clarity:** 4/5 — explanations were well-structured and used the correct vocabulary; genuinely useful when it was right.

</div>


#### Question 2.2.2 — Report the sharpest flaw

What's the single sharpest flaw you found? Quote the bot's exact response (copy-paste, don't paraphrase) and explain in one or two sentences why it's a problem.

<div style="background-color: #f6f8fa; padding: 15px; border-left: 5px solid #0366d6; border-radius: 5px;">

2.2.2.

**Sharpest flaw found:** We asked, "This clade has two subgroups with a short internal branch between them — does that mean they diverged very recently?" The bot responded:

> "Yes, a short internal branch length directly indicates a short time between divergence events — these two subgroups diverged from their common ancestor only a short time before diverging from each other."

**Why this is a problem:** Branch length in most standard phylogenetic tree outputs represents *amount of genetic change* (substitutions per site), not directly elapsed time — the two are only proportional to each other under a strict molecular clock assumption, which the bot never mentioned, checked, or hedged on. It also said nothing about needing the branch's statistical support value before treating the topology itself as reliable. This is a textbook example of a confident, plausible-sounding claim that skips a real caveat a student could easily walk away believing as settled fact.

</div>


#### Question 2.2.3 — Fix it forward

Now that you've seen how your own bot broke (the other group will have reported back to you), what is the *first* thing you'd change if you had five more minutes? Be specific about whether the fix belongs in the system prompt, the knowledge base, or the personality/tone.

<div style="background-color: #f6f8fa; padding: 15px; border-left: 5px solid #0366d6; border-radius: 5px;">

2.2.3.

The other group reported that our RNA-seq Troubleshooting Buddy, when asked a vague question with no error message attached, sometimes guessed at a likely cause instead of insisting on the exact error text first — technically violating our own system-prompt rule 1.

**First fix with 5 more minutes:** Strengthen rule 1 in the system prompt from a general instruction into an explicit hard gate — e.g., "If the student has not provided the exact error message and design formula, your only response is to ask for them; do not speculate about the cause even partially." The current wording left room for the model to be "helpful" by guessing anyway; making the rule un-skippable is a system-prompt fix, not a knowledge-base or personality fix.

</div>


<div style="background-color: #fef9e7; padding: 15px; border-left: 5px solid #d4ac0d; border-radius: 5px;">

**Scoring, timing & wrap-up prep**

Part 2.1 (35 min): roughly 10 min per step 1–3, ~5 min for testing — groups tend to under-invest in step 2.1.4 (actual testing) in favor of over-polishing the system prompt; enforce a hard cutoff so every group has a working bot to swap by minute 35. Part 2.2 (35 min): ~5 min to receive and orient to the swapped bot, ~20 min actively trying to break it, ~10 min writing up 2.2.1-2.2.3. For wrap-up (16:30-17:00), ask each group to prepare a 60-second live demo: show the flaw they found in the bot they attacked, AND the fix they'd make to their own bot. This keeps the wrap-up format identical across groups and fast to run. Rubric scores are meant to prompt honest critique, not to formally grade the bots against each other — frame it that way if competitiveness starts to get in the way of genuinely useful feedback.

</div>

## Glossary

| **Term** | **Definition** |
|---|---|
| **Token / Attention / Transformer** | Quick recap from this morning's reading (Section 7): a transformer breaks text into small chunks called tokens, and uses "attention" to figure out which earlier tokens matter most for predicting the next one. This is the machinery underneath every tool used in this workshop — ChatGPT, Perplexity, OpenEvidence, and QED Science all generate their answers this same way. What differs between them is what each tool is given to work from before it starts generating (see "Retrieval-grounded," below). |
| **Retrieval-grounded** | A tool that looks something up in real documents *before* answering, instead of only generating an answer from what it memorized during training. Perplexity and OpenEvidence are retrieval-grounded; plain ChatGPT with browsing turned off is not. |
| **RAG (Retrieval-Augmented Generation)** | The technical name for the retrieval-grounded technique above: a model looks up relevant external documents before generating its answer, rather than relying only on what it learned during training. This is how tools like Perplexity and OpenEvidence attach citations to their answers. |
| **Hallucination** | When an AI model states something confidently that is false or unsupported — not a lie in the human sense, just a statistically plausible-sounding output that happens to be wrong. |
| **Grounding** | The degree to which a model's answer is tied to verifiable external sources, rather than generated purely from its internal training. A well-grounded answer can be checked; a poorly-grounded one often cannot. |
| **Prompt injection** | An attempt to make a chatbot ignore its original instructions by embedding new, conflicting instructions inside the conversation or input it processes — a common way to "break" a custom bot. |
| **System prompt** | The instructions given to a chatbot before the user ever sees it, defining its purpose, tone, and behavioral rules. Importantly, writing a system prompt does **not** change the model's underlying trained weights (see reading, Section 5) — it's new instructions for the same underlying model, not retraining. This is what you wrote in Question 2.1.2. |
| **Custom GPT** | A version of ChatGPT customized with a specific system prompt, knowledge base, and behavior for a defined purpose — what you built in Part 2.1. |
| **MCP (Model Context Protocol)** | A standard that lets an AI model connect to and query external tools or databases directly (e.g., a drug database) rather than relying only on what it was trained on — mentioned in this morning's lecture demo. |
| **Evidence-grounded AI** | An AI tool designed to answer only from a defined, trusted set of sources (e.g., peer-reviewed journals) rather than the open web — OpenEvidence is an example. |



In [ ]:
#Run this code block
# AI Usage Declaration -- do not edit this cell.
from IPython.display import Javascript, display
display(Javascript("$('div.jp-Cell-inputWrapper').hide();"))

import sys
sys.path.append("../../student_version")  # TODO: verify this relative path matches this notebook's location in your repo
import ai_tracker

ai_tracker.display_AI_decleration()
